# Export du modèle champion pour le projet 8

## Objectif

Exporter ce modèle en `.joblib` pour le rendre indépendant de MLflow au moment du déploiement. En production (dans le conteneur Docker), on ne veut pas dépendre d'un serveur MLflow juste pour charger le modèle.

In [1]:
# Imports
import mlflow
import pandas as pd

mlflow.set_tracking_uri("http://127.0.0.1:5000")

c:\Users\jonat\Desktop\Cours en ligne\Formation AI Engineer\Projets\06\Projet\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Étape 1 — Charger le modèle depuis MLflow

Il faut deux choses :
* `mlflow.sklearn.load_model()` avec l'URI `"models:/Credit_Scoring_Model@champion"`
* S’assurer que le tracking URI pointe bien vers le dossier local (normalement c'est automatique si on lance le script depuis le même dossier)

In [2]:
# Étape 1
champion_uri = "models:/Credit_Scoring_Model@champion"
model = mlflow.sklearn.load_model(champion_uri)

## Étape 2 — Vérifier que le modèle fonctionne

Il faut créer un petit tableau de test avec les 20 features attendues (Il sont dans la synthèse Projet 6) et appeller `predict_proba` dessus. Vérifier qu'on obtien une probabilité entre 0 et 1.

In [3]:
# Étape 2
df = pd.DataFrame(data=[[0.75, 0.28, 0.13, 0.7, 0.16, 5000, 2000, 100000, 27, 1, 1, 0.43, 23, -15000, 150, 50, 0.33, 7, -42, 13]], columns=["EXT_SOURCE_2", "EXT_SOURCE_3", "EXT_SOURCE_1", "BUREAU_BUREAU_DEBT_CREDIT_RATIO_MAX", "APP_PAYMENT_RATE", "AMT_ANNUITY", "DAYS_EMPLOYED", "AMT_GOODS_PRICE", "INSTAL_INSTAL_DAYS_LATE_MAX", "NAME_EDUCATION_TYPE_Higher education", "NAME_FAMILY_STATUS_Married", "PREV_PREV_APP_CREDIT_RATIO_MEAN", "POS_NB_ENTRIES", "DAYS_BIRTH", "PREV_DAYS_LAST_DUE_1ST_VERSION_MAX", "POS_CNT_INSTALMENT_FUTURE_MEAN", "INSTAL_INSTAL_PAYMENT_RATIO_MEAN", "PREV_NAME_CONTRACT_STATUS_REFUSED_MEAN", "BUREAU_DAYS_ENDDATE_FACT_MAX", "INSTAL_NB_PAYMENTS"])

y_pred_proba = model.predict_proba(df)[0][1]

print(f"Probabilité : {y_pred_proba}")

c:\Users\jonat\Desktop\Cours en ligne\Formation AI Engineer\Projets\06\Projet\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Probabilité : 0.5264957353165355


## Étape 3 — Sauvegarder avec joblib

Regarder du côté de `joblib.dump()` pour exporter le pipeline. Puis le recharger avec `joblib.load()` et vérifier qu'on obtien exactement les mêmes résultats qu'avec la version MLflow.

In [5]:
# Étape 3
import joblib

# Save model
joblib.dump(model, 'model.joblib')

# Reload local model for testing
model_local = joblib.load('model.joblib') # Load model
y_pred_proba_local = model_local.predict_proba(df)[0][1] # Predict proba
print(f"Probabilité : {y_pred_proba_local}")

Probabilité : 0.5264957353165355


c:\Users\jonat\Desktop\Cours en ligne\Formation AI Engineer\Projets\06\Projet\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Conclusion

Les deux modèles, extrait de MLflow et sauvegardé ont la même probabilité de prédiction, donc ils se comportent pareil.